## Requirements
- Chrome 116+ (Document PiP) for true always-on-top windows.
- Fallback to standard popup windows if PiP is unavailable.
- Sync note content via localStorage + BroadcastChannel.

In [ ]:
// Floating Notes via Document Picture-in-Picture
const NOTES_CHANNEL = 'ts-floating-notes';
const STORAGE_KEY_PREFIX = 'ts:note:';
function storageKey(id) { return `${STORAGE_KEY_PREFIX}${id}`; }

export async function openFloatingNote(noteId = 'quick') {
  const supportsDocPiP = typeof documentPictureInPicture !== 'undefined' &&
    typeof documentPictureInPicture.requestWindow === 'function';

  if (!supportsDocPiP) {
    console.warn('Document PiP not supported; falling back to popup.');
    return openPopupNote(noteId);
  }

  const pip = await documentPictureInPicture.requestWindow({ width: 420, height: 320 });
  const doc = pip.document;
  doc.body.style.margin = '0';
  doc.body.style.background = '#0b0f1a';
  doc.body.style.fontFamily = 'Inter, system-ui, sans-serif';

  const container = doc.createElement('div');
  container.style.cssText = [
    'display:flex',
    'flex-direction:column',
    'gap:8px',
    'padding:12px',
    'width:100%',
    'height:100%',
    'box-sizing:border-box'
  ].join(';');

  const header = doc.createElement('div');
  header.textContent = 'TradeScout Note';
  header.style.cssText = 'color:#ffb26b;font-weight:600;font-size:13px';

  const textarea = doc.createElement('textarea');
  textarea.placeholder = 'Type your note…';
  textarea.style.cssText = [
    'flex:1',
    'width:100%',
    'resize:none',
    'border-radius:12px',
    'border:1px solid #334155',
    'background:#0f172a',
    'color:#e2e8f0',
    'padding:12px',
    'font-size:13px',
    'outline:none'
  ].join(';');

  const footer = doc.createElement('div');
  footer.style.cssText = 'display:flex;gap:8px;justify-content:space-between;align-items:center';
  const status = doc.createElement('span');
  status.style.cssText = 'color:#94a3b8;font-size:12px';
  status.textContent = 'Saved';

  const closeBtn = doc.createElement('button');
  closeBtn.textContent = 'Close';
  closeBtn.style.cssText = [
    'border:none',
    'border-radius:999px',
    'padding:8px 12px',
    'background:#334155',
    'color:#e2e8f0',
    'font-size:12px',
    'cursor:pointer'
  ].join(';');
  closeBtn.addEventListener('click', () => pip.close());

  footer.append(status, closeBtn);
  container.append(header, textarea, footer);
  doc.body.append(container);

  // Load initial text
  const initial = localStorage.getItem(storageKey(noteId)) ?? '';
  textarea.value = initial;

  // Sync via BroadcastChannel
  const channel = new BroadcastChannel(NOTES_CHANNEL);
  const postUpdate = (text) => channel.postMessage({ type: 'update', id: noteId, text });

  const save = (text) => {
    try { localStorage.setItem(storageKey(noteId), text); } catch {}
    status.textContent = 'Saved';
    postUpdate(text);
  };

  textarea.addEventListener('input', () => {
    status.textContent = 'Saving…';
    save(textarea.value);
  });

  channel.addEventListener('message', (ev) => {
    const data = ev.data || {};
    if (data.type === 'update' && data.id === noteId) {
      if (doc.activeElement !== textarea) {
        textarea.value = data.text || '';
        status.textContent = 'Synced';
      }
    }
  });

  pip.addEventListener('pagehide', () => channel.close());

  return pip;
}

In [ ]:
// Fallback to popup window when Document PiP is unavailable
export function openPopupNote(noteId = 'quick') {
  const w = window.open('', `ts-note-${noteId}`, 'width=420,height=320,menubar=no,toolbar=no,location=no,status=no');
  if (!w) { alert('Popup blocked. Allow popups for floating notes.'); return; }
  const doc = w.document;
  doc.write('<!doctype html><title>TradeScout Note</title>');
  const style = doc.createElement('style');
  style.textContent = `
    body{margin:0;background:#0b0f1a;font-family:Inter,system-ui,sans-serif}
    .wrap{display:flex;flex-direction:column;gap:8px;padding:12px;height:100vh;box-sizing:border-box}
    h1{color:#ffb26b;font-weight:600;font-size:13px;margin:0}
    textarea{flex:1;width:100%;resize:none;border-radius:12px;border:1px solid #334155;background:#0f172a;color:#e2e8f0;padding:12px;font-size:13px;outline:none}
    .footer{display:flex;gap:8px;justify-content:space-between;align-items:center}
    .status{color:#94a3b8;font-size:12px}
    button{border:none;border-radius:999px;padding:8px 12px;background:#334155;color:#e2e8f0;font-size:12px;cursor:pointer}
  `;
  doc.head.append(style);
  const wrap = doc.createElement('div'); wrap.className = 'wrap';
  const h1 = doc.createElement('h1'); h1.textContent = 'TradeScout Note';
  const ta = doc.createElement('textarea'); ta.placeholder = 'Type your note…';
  const footer = doc.createElement('div'); footer.className = 'footer';
  const status = doc.createElement('span'); status.className = 'status'; status.textContent = 'Saved';
  const closeBtn = doc.createElement('button'); closeBtn.textContent = 'Close'; closeBtn.onclick = () => w.close();
  footer.append(status, closeBtn); wrap.append(h1, ta, footer); doc.body.append(wrap);

  const key = `${STORAGE_KEY_PREFIX}${noteId}`;
  ta.value = localStorage.getItem(key) ?? '';
  const channel = new BroadcastChannel(NOTES_CHANNEL);
  ta.addEventListener('input', () => {
    try { localStorage.setItem(key, ta.value); } catch {}
    status.textContent = 'Saved';
    channel.postMessage({ type: 'update', id: noteId, text: ta.value });
  });
  channel.addEventListener('message', (ev) => {
    const d = ev.data || {};
    if (d.type === 'update' && d.id === noteId && doc.activeElement !== ta) {
      ta.value = d.text || ''; status.textContent = 'Synced';
    }
  });
  w.addEventListener('beforeunload', () => channel.close());
  return w;
}

## Usage
- Call `openFloatingNote('quick')` from any UI button or command palette.
- Multiple notes can be keyed: `openFloatingNote('project-123')`.
- Content persists per key in localStorage and syncs across windows via BroadcastChannel.
- Document PiP windows stay on top of other applications; popups generally do not.

## Integration Plan
- Add a "Floating Notes" action in the Scout header or command menu.
- Wrap calls in a small tool function (e.g., `src/agent/tools/notes.ts`).
- Track usage via a lightweight analytics hook.
- Gracefully fall back to popups with a user hint to allow popups.